# Домашнее задание 1. Бандиты, MDP и метод Cross-Entropy

**Вес в оценке:** базовое ДЗ, среднее по сложности.

Задание состоит из трёх частей:

1. **Практика: бандит-алгоритмы (50 баллов)**
2. **Теория: return, уравнение Беллмана, марковские цепи и MDP (30 баллов)**
3. **Практика: метод Cross-Entropy на Frozen Lake 8×8 (20 баллов)**

Части задания с `assert` проверяются автоматически при запуске ячейки — если assert не
упал, эта часть засчитана. Текстовые ответы и графики проверяются вручную.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

rng = np.random.default_rng(0)

## Часть 1. Практика: бандит-алгоритмы (50 баллов)

### 1.1 ε-greedy с убывающим ε (15 баллов)

На семинаре мы реализовали ε-greedy с постоянным ε. Проблема: даже когда агент уверен
в лучшей руке, он продолжает исследовать с той же частотой ε, из-за чего regret растёт
линейно, а не логарифмически.

**Задание:** реализуйте `DecayingEpsilonGreedyAgent`, где ε убывает со временем,
например ε_t = min(1, c / (t + 1)) для некоторой константы c, или экспоненциальное затухание.
Класс должен иметь тот же интерфейс, что и агенты с семинара: `select_arm()`, `update(arm, reward)`.

In [ ]:
class DecayingEpsilonGreedyAgent:
    def __init__(self, n_arms, c=1.0, rng=None):
        self.n_arms = n_arms
        self.c = c
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms)
        self.t = 0
        self.rng = rng or np.random.default_rng()

    def _epsilon(self) -> float:
        # TODO: реализуйте расписание epsilon_t, например min(1, c / (t+1))
        raise NotImplementedError

    def select_arm(self) -> int:
        # TODO
        raise NotImplementedError

    def update(self, arm: int, reward: float):
        # TODO: такое же инкрементальное обновление Q, как у EpsilonGreedyAgent на семинаре
        raise NotImplementedError

In [ ]:
# Самопроверка: агент должен постепенно приближаться к чисто жадному поведению
_agent = DecayingEpsilonGreedyAgent(n_arms=3, c=5.0, rng=np.random.default_rng(1))
for _ in range(1000):
    a = _agent.select_arm()
    assert 0 <= a < 3
    _agent.update(a, reward=float(a == 1))

assert _agent._epsilon() < 0.1, "после 1000 шагов epsilon должен стать маленьким"
print("OK: DecayingEpsilonGreedyAgent проходит базовую проверку")

### 1.2 Сравнение на нескольких конфигурациях (20 баллов)

На семинаре мы сравнивали агентов на одной конфигурации рук. Постройте сравнение
(средний накопленный regret) для **трёх** разных конфигураций `bandit_probs`:

1. «Лёгкая» — руки сильно отличаются, например `[0.1, 0.9]`
2. «Сложная» — руки почти одинаковые, например `[0.45, 0.5, 0.48]`
3. «Много рук» — например 10 рук со случайными вероятностями

Сравните **3 агентов**: `EpsilonGreedyAgent` (константный ε), `DecayingEpsilonGreedyAgent`,
`UCB1Agent` (используйте реализации с семинара — скопируйте их сюда или импортируйте,
если оформили семинар как модуль).

Постройте 3 графика (по одному на конфигурацию), на каждом — 3 кривые regret.

In [ ]:
# Скопируйте сюда классы BernoulliBanditEnv, EpsilonGreedyAgent, UCB1Agent
# и функцию run_agent из семинара, либо импортируйте их.

# TODO: ваш код здесь

**Вопрос (ответьте текстом в этой ячейке):** на какой из трёх конфигураций разница между
UCB1 и ε-greedy наиболее заметна, и почему? Что происходит с «лёгкой»
конфигурацией при увеличении числа рук с явно плохими вероятностями?

_Ваш ответ:_ TODO

### 1.3 Устойчивость к гиперпараметрам (15 баллов)

Постройте график зависимости **итогового** (на последнем шаге) среднего regret от
гиперпараметра:

* для ε-greedy — от константного ε ∈ {0.01, 0.05, 0.1, 0.2, 0.5}
* для UCB1 — от c ∈ {0.5, 1, 2, 4, 8}

Сделайте вывод о том, насколько результат чувствителен к выбору гиперпараметра
у каждого метода.

In [ ]:
# TODO: ваш код здесь

### 1.4 (бонус, 15 баллов) Non-stationary бандит

Что если вероятности рук **меняются со временем**? Реализуйте среду, где `probs` дрейфуют
(например, раз в 500 шагов случайно перемешиваются, или на каждом шаге добавляется малый
гауссовский шум с последующим клипом в [0, 1]).

Сравните на такой среде обычное усреднение Q (как в `EpsilonGreedyAgent`) с
**экспоненциально взвешенным** усреднением (constant step-size: `Q += alpha * (r - Q)`
вместо `Q += (r - Q) / N`). Какой вариант лучше отслеживает изменения среды и почему?

In [ ]:
# TODO (бонус): ваш код здесь

## Часть 2. Теория (30 баллов)

Отвечайте прямо в markdown-ячейках, используя LaTeX ($...$ или $$...$$). Показывайте
промежуточные шаги, а не только финальный ответ — за них тоже начисляются баллы.

### 2.1 Return для простого эпизода (5 баллов)

Дан эпизод (последовательность наград после каждого шага):

$$
R_0 = 1,\; R_1 = 0,\; R_2 = 2,\; R_3 = 0,\; R_4 = 1 \quad \text{(эпизод завершается после шага 4)}
$$

Посчитайте $G_0, G_1, \ldots, G_4$ для $\gamma = 0.9$. Покажите, как использовать
рекуррентное соотношение $G_t = R_t + \gamma G_{t+1}$, чтобы не считать каждую
сумму с нуля.

_Ваш ответ:_ TODO

### 2.2 Марковская цепь (5 баллов)

Погода в городе описывается марковской цепью с состояниями {Солнце, Облачно, Дождь} и матрицей переходов

$$
P = \begin{pmatrix} 0.7 & 0.2 & 0.1 \\ 0.3 & 0.4 & 0.3 \\ 0.2 & 0.3 & 0.5 \end{pmatrix}.
$$

1. Сегодня солнце. Какова вероятность дождя послезавтра? Запишите вычисление через $p_0 P^2$.
2. Найдите стационарное распределение $\pi^\top P = \pi^\top$ (аналитически или численно в ячейке ниже — но запишите систему уравнений).
3. Проверьте численно: возведите $P$ в большую степень и сравните строки с найденным $\pi$.

_Ваш ответ:_ TODO

In [ ]:
P_weather = np.array([[0.7, 0.2, 0.1],
                      [0.3, 0.4, 0.3],
                      [0.2, 0.3, 0.5]])
# TODO: p_0 @ P^2, стационарное распределение, проверка через np.linalg.matrix_power

### 2.3 Вывод уравнения Беллмана для $V^\pi$ (10 баллов)

Исходя из определения $V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]$ и рекуррентного
соотношения $G_t = R_t + \gamma G_{t+1}$, выведите уравнение Беллмана:

$$
V^\pi(s) = \sum_a \pi(a|s) \Big[ R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V^\pi(s') \Big]
$$

Распишите каждый шаг вывода (где именно используется линейность матожидания,
где — марковское свойство, где — определение $\pi$ и $P$).

_Ваш ответ:_ TODO

### 2.4 MDP на бумаге: робот-уборщик (5 баллов)

Робот-уборщик находится в одной из трёх комнат: `{Кухня, Зал, Коридор}`. Действия:
`{Убирать, Переехать}`.

* Из **Кухни**: `Убирать` -> остаться в Кухне с наградой +2 (вероятность 1);
  `Переехать` -> Зал с наградой 0 (вероятность 0.8) или Коридор с наградой 0 (вероятность 0.2)
* Из **Зала**: `Убирать` -> остаться в Зале с наградой +1 (вероятность 1);
  `Переехать` -> Кухня с наградой 0 (вероятность 0.5) или Коридор с наградой 0 (вероятность 0.5)
* Из **Коридора**: `Убирать` -> остаться в Коридоре с наградой 0 (в коридоре убирать нечего);
  `Переехать` -> Кухня с наградой 0 (вероятность 0.5) или Зал с наградой 0 (вероятность 0.5)

Пусть политика π детерминированная: `Убирать` в Кухне и Зале, `Переехать` в Коридоре.
Возьмите γ = 0.9.

**Задание:** запишите систему из 3 линейных уравнений на $V^\pi(\text{Кухня})$,
$V^\pi(\text{Зал})$, $V^\pi(\text{Коридор})$ по уравнению Беллмана из 2.3, и решите её
(аналитически или через `numpy.linalg.solve` в коде ниже — но составить систему нужно руками).

_Ваш ответ (система уравнений):_ TODO

In [ ]:
# Опционально: проверьте свой аналитический ответ численно.
# Задача сводится к V = R_pi + gamma * P_pi @ V, то есть (I - gamma * P_pi) @ V = R_pi.

gamma = 0.9
# P_pi = ...
# R_pi = ...
# V = np.linalg.solve(np.eye(3) - gamma * P_pi, R_pi)
# print(V)

### 2.5 Сформулируйте как MDP (5 баллов)

Для **двух** задач из списка опишите MDP: множество состояний (что агент знает, чего не знает),
действия, переходы и **где в них случайность**, награда за шаг, эпизод (или задача непрерывная).

* Лифт в 10-этажном доме, который решает, куда ехать.
* Игра «2048».
* Полив теплицы: датчики влажности и температуры, насос.
* Любая задача из вашей жизни.

_Ваш ответ:_ TODO

## Часть 3. Метод Cross-Entropy на Frozen Lake 8×8 (20 баллов)

На семинаре мы обучили табличный Cross-Entropy на озере 4×4. Карта 8×8 (`map_name="8x8"`)
сложнее: 64 состояния, до цели далеко, случайная политика доходит крайне редко.

**Задание:**

1. (10 баллов) Реализуйте `train_cem` (можно взять за основу код семинара) и обучите агента на
   `FrozenLake-v1`, `map_name="8x8"`, `is_slippery=False`. Постройте кривую обучения (средний return по итерациям).
   Самопроверка ниже требует **долю успехов ≥ 90%** у выученной (стохастической) политики. Подсказка: случайная
   политика на 8×8 доходит до цели очень редко, поэтому сессий нужно несколько сотен; помогает сглаживание.
2. (5 баллов) Сравните три варианта: без сглаживания, сглаживание по Лапласу, смешивание со старой политикой.
   Какой быстрее, какой стабильнее (по 3 запускам с разными сидами)?
3. (5 баллов) Ответьте текстом: почему без сглаживания алгоритм иногда «зависает» на нулевом return,
   и что именно чинит сглаживание по Лапласу? Почему **жадная** версия выученной политики (`argmax` в каждой
   клетке) иногда не доходит до цели, хотя стохастическая доходит всегда? Что произойдёт на скользком льду и почему?

In [ ]:
def train_cem(env, n_iter=40, n_sessions=300, q=0.7, laplace=0.0, mix=1.0, seed=0):
    # TODO: верните (policy, history), где policy — массив n_states x n_actions,
    #       history — список средних return по итерациям
    raise NotImplementedError

In [ ]:
env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)
policy, history = train_cem(env, seed=0)

plt.plot(history, marker="o")
plt.xlabel("итерация"); plt.ylabel("средний return")
plt.title("Cross-Entropy на FrozenLake 8x8")
plt.show()

def policy_success_rate(env, policy, n_episodes=200, seed=123):
    eval_rng = np.random.default_rng(seed)
    wins = 0
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep)
        while True:
            a = int(eval_rng.choice(policy.shape[1], p=policy[obs]))
            obs, r, terminated, truncated, _ = env.step(a)
            if terminated or truncated:
                wins += r > 0
                break
    return wins / n_episodes

_rate = policy_success_rate(env, policy)
print(f"доля успехов выученной политики: {_rate:.1%}")
assert policy.shape == (64, 4), "политика должна быть таблицей 64 x 4"
assert _rate >= 0.9, "нужно доходить до цели не реже чем в 90% эпизодов"
print("OK: Cross-Entropy на 8x8 засчитан")
env.close()

In [ ]:
# TODO: сравнение трёх вариантов сглаживания (задание 2)

_Ответ на вопрос 3:_ TODO

## Чек-лист перед сдачей

- [ ] `DecayingEpsilonGreedyAgent` реализован и проходит self-check
- [ ] Построено сравнение 3 агентов на 3 конфигурациях бандита, отвечен вопрос про разницу между конфигурациями
- [ ] Построен график чувствительности к гиперпараметрам
- [ ] Посчитан return $G_0, \ldots, G_4$
- [ ] Решена задача про марковскую цепь погоды
- [ ] Выведено уравнение Беллмана для $V^\pi$ по шагам
- [ ] Составлена и решена система уравнений для робота-уборщика
- [ ] Две задачи сформулированы как MDP
- [ ] `train_cem` проходит self-check на 8×8, есть сравнение вариантов сглаживания и ответ на вопрос
- [ ] (опционально) реализован non-stationary бандит и сравнение способов усреднения